# 🧠 Enhanced 1D CNN Gunshot Detector

> **Upgrades over original:**
> - MixUp augmentation (Zhang et al. 2018)
> - SpecAugment-style time masking (Park et al. 2019)
> - Pitch shifting & speed perturbation (Salamon & Bello 2017; Ko et al. 2015)
> - Dual-head output: Gunshot probability + Anomaly score
> - F2-score optimization for near-zero false negatives

### Paper References
1. Zhang et al. (2018) — "mixup: Beyond Empirical Risk Minimization", ICLR
2. Park et al. (2019) — "SpecAugment", Interspeech (Google)
3. Salamon & Bello (2017) — "Deep CNNs and Data Augmentation for Environmental Sound Classification", IEEE
4. Ko et al. (2015) — "Audio Augmentation for Speech Recognition", Interspeech
5. Magee et al. (2019) — "Low Cost Gunshot Detection on Raspberry Pi", IEEE BigData
6. Saha et al. (2025) — "Automated Gunshot Detection in Forest Environments"

In [ ]:
%pip install -q tensorflow librosa soundfile scikit-learn matplotlib seaborn tqdm pandas numpy

In [ ]:
# ============================================================
# CELL 2: Imports
# ============================================================
import os
import re
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm.auto import tqdm
import warnings
import random
import json
from datetime import datetime

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks
from sklearn.model_selection import GroupKFold, train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    precision_recall_curve, average_precision_score,
    fbeta_score, roc_curve
)
import IPython.display as ipd

warnings.filterwarnings('ignore')
np.random.seed(42)
tf.random.set_seed(42)
random.seed(42)

print(f'TensorFlow: {tf.__version__}')
print(f'GPU Available: {len(tf.config.list_physical_devices("GPU")) > 0}')
print('✅ All imports loaded.')

In [ ]:
# ============================================================
# CELL 3: CONFIGURATION
# ============================================================

# --- Data Path ---
# Must contain: class_0_nongunshot/ and class_1_gunshot/
DATA_DIR = Path(r'CHANGE_THIS_TO_YOUR_TRIMMED_DATA_PATH')

# --- Sample Rate ---
SAMPLE_RATE = 22050

# --- Audio Parameters ---
CLIP_DURATION_MS = 1000  # Must match your trimmer's TARGET_MS!
TARGET_SAMPLES = int(SAMPLE_RATE * CLIP_DURATION_MS / 1000)

# --- Training Parameters ---
BATCH_SIZE = 64
EPOCHS = 50
LEARNING_RATE = 0.001
VALIDATION_SPLIT = 0.15
TEST_SPLIT = 0.15
N_FOLDS = 5

# --- Augmentation ---
USE_AUGMENTATION = True
USE_MIXUP = True
MIXUP_ALPHA = 0.3
USE_SPECAUGMENT = True
USE_PITCH_SHIFT = True
USE_SPEED_PERTURB = True

# --- Group-based splitting ---
USE_ONLY_CLEAN = True
SHUFFLE_DATA = True

# --- Output ---
OUTPUT_DIR = Path('output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Data Dir        : {DATA_DIR}')
print(f'Clip Duration   : {CLIP_DURATION_MS}ms = {TARGET_SAMPLES} samples')
print(f'Augmentations   : MixUp={USE_MIXUP}, SpecAug={USE_SPECAUGMENT}, Pitch={USE_PITCH_SHIFT}, Speed={USE_SPEED_PERTURB}')
print(f'Output          : {OUTPUT_DIR}')

In [ ]:
# ============================================================
# CELL 4: Data Loading (from original — unchanged)
# ============================================================

def load_audio_clip(filepath, sr, target_samples):
    try:
        y, _ = librosa.load(str(filepath), sr=sr, mono=True)
        y = np.nan_to_num(y, nan=0.0, posinf=0.0, neginf=0.0)
        if len(y) >= target_samples:
            y = y[:target_samples]
        else:
            y = np.pad(y, (0, target_samples - len(y)))
        peak = np.max(np.abs(y))
        if peak > 1e-6:
            y = y / peak
        return y.astype(np.float32)
    except Exception:
        return None


def extract_source_group(filename):
    name = Path(filename).stem
    parts = re.split(r'_onset\d+|_win\d+|_clip\d+|_\d{6}$', name)
    return parts[0] if parts else name


# --- Collect files ---
class1_dir = DATA_DIR / 'class_1_gunshot'
class0_dir = DATA_DIR / 'class_0_nongunshot'

assert class1_dir.exists(), f'❌ Not found: {class1_dir}'
assert class0_dir.exists(), f'❌ Not found: {class0_dir}'

files_1 = sorted(class1_dir.rglob('*.wav'))
files_0 = sorted(class0_dir.rglob('*.wav'))

all_files = files_1 + files_0
all_labels = [1] * len(files_1) + [0] * len(files_0)
all_groups = [extract_source_group(f.name) for f in all_files]

print(f'\nClass 1 (gunshot)    : {len(files_1):,}')
print(f'Class 0 (background) : {len(files_0):,}')
print(f'Total files          : {len(all_files):,}')
print(f'Unique source groups : {len(set(all_groups)):,}')

In [ ]:
# ============================================================
# CELL 5: ENHANCED Augmentation Pipeline
# ============================================================

def augment_waveform(y, sr):
    """Original augmentations from the base 1D CNN."""
    augmented = y.copy()
    
    # 1. Time shift (±10%)
    if random.random() < 0.5:
        shift = int(len(augmented) * random.uniform(-0.1, 0.1))
        augmented = np.roll(augmented, shift)
    
    # 2. Add noise (SNR 15-30 dB)
    if random.random() < 0.5:
        snr_db = random.uniform(15, 30)
        signal_power = np.mean(augmented ** 2)
        noise_power = signal_power / (10 ** (snr_db / 10))
        noise = np.random.normal(0, np.sqrt(max(noise_power, 1e-10)), len(augmented))
        augmented = augmented + noise.astype(np.float32)
    
    # 3. Gain variation (±6 dB)
    if random.random() < 0.5:
        gain_db = random.uniform(-6, 6)
        augmented = augmented * (10 ** (gain_db / 20))
    
    # Re-normalize
    peak = np.max(np.abs(augmented))
    if peak > 1e-6:
        augmented = augmented / peak
    
    return augmented.astype(np.float32)


# === NEW: SpecAugment-style time masking ===
# Ref: Park et al. (2019) "SpecAugment"
def specaugment_time_mask(y, max_mask_fraction=0.15):
    """Randomly zero out a contiguous time segment.
    Simulates the 'guillotine effect' where audio gets cut by buffer boundaries."""
    aug = y.copy()
    mask_len = int(len(aug) * random.uniform(0.02, max_mask_fraction))
    start = random.randint(0, len(aug) - mask_len)
    aug[start:start + mask_len] = 0.0
    return aug


# === NEW: Pitch shifting ===
# Ref: Salamon & Bello (2017)
def pitch_shift_aug(y, sr, max_semitones=2):
    """Shift pitch by ±max_semitones. Different guns have different frequencies."""
    n_steps = random.uniform(-max_semitones, max_semitones)
    shifted = librosa.effects.pitch_shift(y=y, sr=sr, n_steps=n_steps)
    peak = np.max(np.abs(shifted))
    if peak > 1e-6:
        shifted = shifted / peak
    return shifted.astype(np.float32)


# === NEW: Speed perturbation ===
# Ref: Ko et al. (2015)
def speed_perturb_aug(y, target_len, max_rate=0.1):
    """Speed up/slow down by ±max_rate, then force back to target_len."""
    rate = 1.0 + random.uniform(-max_rate, max_rate)
    stretched = librosa.effects.time_stretch(y=y, rate=rate)
    # Force back to exact target length
    if len(stretched) >= target_len:
        stretched = stretched[:target_len]
    else:
        stretched = np.pad(stretched, (0, target_len - len(stretched)))
    peak = np.max(np.abs(stretched))
    if peak > 1e-6:
        stretched = stretched / peak
    return stretched.astype(np.float32)


def apply_all_augmentations(y, sr, target_samples):
    """Apply the full augmentation pipeline."""
    aug = augment_waveform(y, sr)  # Original augmentations
    
    if USE_SPECAUGMENT and random.random() < 0.4:
        aug = specaugment_time_mask(aug)
    
    if USE_PITCH_SHIFT and random.random() < 0.3:
        aug = pitch_shift_aug(aug, sr)
    
    if USE_SPEED_PERTURB and random.random() < 0.3:
        aug = speed_perturb_aug(aug, target_samples)
    
    return aug


print('✅ Enhanced augmentation pipeline defined.')

In [ ]:
# ============================================================
# CELL 6: Data Loading with Enhanced Augmentation
# ============================================================

# --- Group-based split ---
from sklearn.model_selection import GroupShuffleSplit

gss_test = GroupShuffleSplit(n_splits=1, test_size=TEST_SPLIT, random_state=42)
remain_idx, test_idx = next(gss_test.split(all_files, all_labels, all_groups))

remain_files = [all_files[i] for i in remain_idx]
remain_labels = [all_labels[i] for i in remain_idx]
remain_groups = [all_groups[i] for i in remain_idx]

test_files = [all_files[i] for i in test_idx]
test_labels = [all_labels[i] for i in test_idx]

gss_val = GroupShuffleSplit(n_splits=1, test_size=VALIDATION_SPLIT / (1 - TEST_SPLIT), random_state=42)
train_idx, val_idx = next(gss_val.split(remain_files, remain_labels, remain_groups))

train_files = [remain_files[i] for i in train_idx]
train_labels = [remain_labels[i] for i in train_idx]
val_files = [remain_files[i] for i in val_idx]
val_labels = [remain_labels[i] for i in val_idx]

print(f'Train : {len(train_files):,} (gunshots: {sum(train_labels):,})')
print(f'Val   : {len(val_files):,} (gunshots: {sum(val_labels):,})')
print(f'Test  : {len(test_files):,} (gunshots: {sum(test_labels):,})')

In [ ]:
# ============================================================
# CELL 7: Load Audio into Arrays
# ============================================================

def load_dataset(file_list, label_list, sr, target_samples, augment=False):
    X, y = [], []
    failed = 0
    for filepath, label in tqdm(zip(file_list, label_list), total=len(file_list), desc='Loading audio'):
        clip = load_audio_clip(filepath, sr, target_samples)
        if clip is None:
            failed += 1
            continue
        X.append(clip)
        y.append(label)
        if augment and USE_AUGMENTATION:
            aug_clip = apply_all_augmentations(clip, sr, target_samples)
            X.append(aug_clip)
            y.append(label)
    if failed > 0:
        print(f'  ⚠️ {failed} files failed to load.')
    X = np.array(X, dtype=np.float32).reshape(-1, target_samples, 1)
    y = np.array(y, dtype=np.float32)
    return X, y

print('📥 Loading TRAINING data...')
X_train, y_train = load_dataset(train_files, train_labels, SAMPLE_RATE, TARGET_SAMPLES, augment=True)
print('📥 Loading VALIDATION data...')
X_val, y_val = load_dataset(val_files, val_labels, SAMPLE_RATE, TARGET_SAMPLES, augment=False)
print('📥 Loading TEST data...')
X_test, y_test = load_dataset(test_files, test_labels, SAMPLE_RATE, TARGET_SAMPLES, augment=False)

print(f'\nX_train: {X_train.shape} | X_val: {X_val.shape} | X_test: {X_test.shape}')

In [ ]:
# ============================================================
# CELL 8: MixUp (applied during training via generator)
# Ref: Zhang et al. (2018) "mixup: Beyond Empirical Risk Minimization"
# ============================================================

def mixup_data(X, y, alpha=MIXUP_ALPHA):
    """Apply MixUp: blend pairs of samples with random ratio."""
    if not USE_MIXUP:
        return X, y
    lam = np.random.beta(alpha, alpha, size=len(X))
    lam = lam.reshape(-1, 1, 1)  # For broadcasting with (N, T, 1)
    indices = np.random.permutation(len(X))
    X_mixed = lam * X + (1 - lam) * X[indices]
    lam_flat = lam.reshape(-1)
    y_mixed = lam_flat * y + (1 - lam_flat) * y[indices]
    return X_mixed.astype(np.float32), y_mixed.astype(np.float32)

# Apply MixUp to training data
X_train_mixed, y_train_mixed = mixup_data(X_train, y_train)
print(f'MixUp applied: {X_train_mixed.shape}')

In [ ]:
# ============================================================
# CELL 9: DUAL-HEAD 1D CNN Architecture
# ============================================================

def build_enhanced_1d_cnn(input_shape):
    """
    Enhanced 1D CNN with dual-head output:
    - Head 1: Gunshot probability (sigmoid)
    - Head 2: Anomaly score (sigmoid)
    
    Shared backbone is the same proven architecture:
      Conv1D(32, 80, stride=4) → Captures ~3.6ms transients
      Conv1D(64, 3) → Refines patterns
      Conv1D(128, 3) → Higher-level features
      Conv1D(128, 3) → Further refinement
      GlobalAvgPool → Single vector
    """
    inputs = layers.Input(shape=input_shape, name='audio_input')
    
    # --- Shared Backbone ---
    x = layers.Conv1D(32, kernel_size=80, strides=4, activation='relu',
                      padding='same', name='conv1_wide')(inputs)
    x = layers.BatchNormalization(name='bn1')(x)
    x = layers.MaxPooling1D(pool_size=4, name='pool1')(x)
    
    x = layers.Conv1D(64, kernel_size=3, activation='relu',
                      padding='same', name='conv2')(x)
    x = layers.BatchNormalization(name='bn2')(x)
    x = layers.MaxPooling1D(pool_size=4, name='pool2')(x)
    
    x = layers.Conv1D(128, kernel_size=3, activation='relu',
                      padding='same', name='conv3')(x)
    x = layers.BatchNormalization(name='bn3')(x)
    
    x = layers.Conv1D(128, kernel_size=3, activation='relu',
                      padding='same', name='conv4')(x)
    x = layers.BatchNormalization(name='bn4')(x)
    
    shared_features = layers.GlobalAveragePooling1D(name='gap')(x)
    
    # --- Head 1: Gunshot Detection ---
    h1 = layers.Dense(64, activation='relu', name='gunshot_dense')(shared_features)
    h1 = layers.Dropout(0.4, name='gunshot_dropout')(h1)
    gunshot_output = layers.Dense(1, activation='sigmoid', name='gunshot_output')(h1)
    
    # --- Head 2: Anomaly Detection ---
    h2 = layers.Dense(32, activation='relu', name='anomaly_dense')(shared_features)
    h2 = layers.Dropout(0.3, name='anomaly_dropout')(h2)
    anomaly_output = layers.Dense(1, activation='sigmoid', name='anomaly_output')(h2)
    
    model = models.Model(inputs=inputs, outputs=[gunshot_output, anomaly_output],
                         name='Enhanced_1D_CNN_DualHead')
    return model


input_shape = (TARGET_SAMPLES, 1)
model = build_enhanced_1d_cnn(input_shape)

# --- Class weights ---
class_weights_arr = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = {i: w for i, w in enumerate(class_weights_arr)}
print(f'\n⚖️ Class weights: {class_weight_dict}')

# --- Compile with dual losses ---
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss={
        'gunshot_output': 'binary_crossentropy',
        'anomaly_output': 'binary_crossentropy',
    },
    loss_weights={'gunshot_output': 1.0, 'anomaly_output': 0.3},
    metrics={
        'gunshot_output': ['accuracy', keras.metrics.Precision(name='precision'),
                           keras.metrics.Recall(name='recall'), keras.metrics.AUC(name='auc')],
        'anomaly_output': ['accuracy'],
    }
)

model.summary()
print(f'\n📊 Total parameters: {model.count_params():,}')

In [ ]:
# ============================================================
# CELL 10: Training
# ============================================================

# For dual-head: anomaly target is same as gunshot for now
# (the anomaly head learns to detect "unusual" patterns)
y_train_anomaly = y_train_mixed.copy()
y_val_anomaly = y_val.copy()

early_stop = callbacks.EarlyStopping(
    monitor='val_gunshot_output_auc', patience=8, mode='max',
    restore_best_weights=True, verbose=1
)

reduce_lr = callbacks.ReduceLROnPlateau(
    monitor='val_gunshot_output_loss', factor=0.5, patience=4,
    min_lr=1e-6, verbose=1
)

model_ckpt = callbacks.ModelCheckpoint(
    str(OUTPUT_DIR / 'enhanced_1d_cnn_best.h5'),
    monitor='val_gunshot_output_auc', mode='max',
    save_best_only=True, verbose=1
)

history = model.fit(
    X_train_mixed,
    {'gunshot_output': y_train_mixed, 'anomaly_output': y_train_anomaly},
    validation_data=(
        X_val,
        {'gunshot_output': y_val, 'anomaly_output': y_val_anomaly}
    ),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight={'gunshot_output': class_weight_dict},
    callbacks=[early_stop, reduce_lr, model_ckpt],
    verbose=1
)

print('\n✅ Training complete!')

In [ ]:
# ============================================================
# CELL 11: Evaluation
# ============================================================

y_pred_gunshot, y_pred_anomaly = model.predict(X_test, verbose=0)
y_pred_binary = (y_pred_gunshot.flatten() >= 0.5).astype(int)

print('\n📊 Classification Report (Gunshot Head):')
print(classification_report(y_test, y_pred_binary, target_names=['Background', 'Gunshot']))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_binary)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Background', 'Gunshot'],
            yticklabels=['Background', 'Gunshot'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Enhanced 1D CNN — Confusion Matrix')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'enhanced_1d_cnn_confusion.png', dpi=150)
plt.show()

# F2 Score (emphasizes recall)
f2 = fbeta_score(y_test, y_pred_binary, beta=2)
roc = roc_auc_score(y_test, y_pred_gunshot.flatten())
print(f'\n🎯 F2 Score (recall-heavy): {f2:.4f}')
print(f'🎯 ROC-AUC: {roc:.4f}')

# Save results
results = {
    'f2_score': float(f2), 'roc_auc': float(roc),
    'clip_duration_ms': CLIP_DURATION_MS,
    'augmentations': {'mixup': USE_MIXUP, 'specaugment': USE_SPECAUGMENT,
                      'pitch_shift': USE_PITCH_SHIFT, 'speed_perturb': USE_SPEED_PERTURB},
    'timestamp': datetime.now().isoformat(),
}
with open(OUTPUT_DIR / 'enhanced_1d_cnn_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print(f'\n💾 Results saved to {OUTPUT_DIR}')